# Evaluate bo767 retrieval recall accuracy with NeMo Retriever

To download the bo767 PDF corpus, please refer to [digital_corpora_download.ipynb](https://github.com/NVIDIA/NeMo-Retriever/blob/main/evaluation/digital_corpora_download.ipynb)

### CLI

In [ ]:
%%bash
retriever pipeline run path/to/bo767/pdfs \
  --vdb-kwargs-json '{"uri":"../lancedb","table_name":"bo767"}' \
  --evaluation-mode beir \
  --beir-dataset-name bo767 \
  --quiet

### Python

In [ ]:
from pathlib import Path

from nemo_retriever import create_ingestor
from nemo_retriever.model import VL_EMBED_MODEL
from nemo_retriever.params import EmbedParams, VdbUploadParams
from nemo_retriever.recall.beir import BeirConfig, evaluate_lancedb_beir, resolve_beir_dataset_options

input_path = str(Path("path/to/bo767/pdfs").resolve())
lancedb_uri = str(Path("../lancedb").resolve())
table_name = "bo767"

result = (
    create_ingestor(run_mode="batch")
    .files(input_path)
    .extract()
    .embed(EmbedParams(model_name=VL_EMBED_MODEL))
    .vdb_upload(
        VdbUploadParams(
            vdb_op="lancedb",
            vdb_kwargs={"uri": lancedb_uri, "table_name": table_name, "overwrite": True},
        )
    )
    .ingest()
)

beir = resolve_beir_dataset_options(dataset_name="bo767")

dataset, raw_hits, run, metrics = evaluate_lancedb_beir(
    BeirConfig(
        lancedb_uri=lancedb_uri,
        lancedb_table=table_name,
        embedding_model=VL_EMBED_MODEL,
        loader=beir.loader,
        dataset_name=beir.dataset_name,
        doc_id_field=beir.doc_id_field,
        ks=beir.ks,
    )
)

metrics